# Project 2

In this project, I will use datasets from Uppsala Conflict Data Program (UCDP) and World Bank Group to analyze the relationship between conflict occurences and the growth of GDP per capita in Syria between 1961 and 2023.

Hypothesis 1: The more intense the conflict is in the year of observation, the more damage inflicted on the growth of GDP per capita.<br>
Hypothesis 2: Some types of conflicts bring more harm to the growth of GDP per capita.

For simplicity, this project does not lag the year of GDP per capita by one year behind the conflict year.

# Step 1: read and clean the files
In this step, I read the two datasets into python and clean them. For convenience and simplicity, I only filter out columns that will be used for analysis and sort the years from 1961 to 2023.

In [67]:
import pandas as pd

# import the first conflict dataset
Conflict = pd.read_csv("Conflict.csv")
Conflict.head()

# filter the dataset for conflicts that involved Syria between 1961 and 2023

Syria_Conflict = Conflict[
    (Conflict["location"].str.contains("Syria"))
    & (Conflict["year"] >= 1961)
    & (Conflict["year"] <= 2023)
]

Syria_Conflict = Syria_Conflict.sort_values("year")
Syria_Conflict_Cleaned = Syria_Conflict[
    ["location", "year", "intensity_level", "cumulative_intensity", "type_of_conflict"]
]
Syria_Conflict_Cleaned.head()

Syria_Conflict = Syria_Conflict_Cleaned.rename(
    columns={"location": "Country Name", "year": "Year"}
)

Syria_Conflict.loc[
    Syria_Conflict["Country Name"].str.contains("syria", case=False, na=False),
    "Country Name",
] = "Syria"

Syria_Conflict.head()

,Country Name,Year,intensity_level,cumulative_intensity,type_of_conflict
1506,Syria,1966,1,0,3
1556,Syria,1967,2,1,2
1557,Syria,1973,2,1,2
1507,Syria,1979,1,0,3
1508,Syria,1980,1,0,3


In [73]:
# import the second GDP per capita dataset
GDP = pd.read_csv("GDP.csv")
GDP.head()

# filter the dataset for the annual growth of GDP per capita in Syria between 1961 and 2023

Syria_GDP = GDP[(GDP["Country Name"].str.contains("Syria"))]

# Convert the dataset from wide to long format
Syria_GDP = Syria_GDP.melt(
    id_vars=["Country Name", "Country Code", "Indicator Name", "Indicator Code"],
    var_name="Year",
    value_name="GDP per capita",
)

Syria_GDP.head()

# convert the year column to numeric and drop rows with invalid years (NaN)
Syria_GDP["Year"] = pd.to_numeric(Syria_GDP["Year"], errors="coerce")
Syria_GDP = Syria_GDP.dropna(subset=["Year"])


# filter the dataset for years between 1961 and 2023
Syria_GDP = Syria_GDP[(Syria_GDP["Year"] >= 1961) & (Syria_GDP["Year"] <= 2023)]
Syria_GDP = Syria_GDP.sort_values("Year")
SyriaGDP_Cleaned = Syria_GDP[
    ["Country Name", "Year", "GDP per capita"]
]  # use double brackets or key error


SyriaGDP_Cleaned.loc[
    SyriaGDP_Cleaned["Country Name"].str.contains("syria", case=False, na=False),
    "Country Name",
] = "Syria"

SyriaGDP_Cleaned.head()


,Country Name,Year,GDP per capita
1,Syria,1961,7.565566
2,Syria,1962,20.908845
3,Syria,1963,-11.354072
4,Syria,1964,5.981871
5,Syria,1965,-0.828368


# Step 2: Merge the Datasets
In this step, I merge the two datasets based on year and country name. I then code all the years without conflicts in Syria as 0.

In [78]:
# Merge the two datasets based on year and country name
Merged_Data = pd.merge(
    Syria_Conflict,
    SyriaGDP_Cleaned,
    on=["Country Name", "Year"],
    how="outer",
)

# fill NaN values in the conflict columns with 0
Merged_Data = Merged_Data.fillna(0)

Merged_Data.head(15)

,Country Name,Year,intensity_level,cumulative_intensity,type_of_conflict,GDP per capita
0,Syria,1961,0.0,0.0,0.0,7.565566
1,Syria,1962,0.0,0.0,0.0,20.908845
2,Syria,1963,0.0,0.0,0.0,-11.354072
3,Syria,1964,0.0,0.0,0.0,5.981871
4,Syria,1965,0.0,0.0,0.0,-0.828368
5,Syria,1966,1.0,0.0,3.0,-10.519981
6,Syria,1967,2.0,1.0,2.0,4.870754
7,Syria,1968,0.0,0.0,0.0,0.457908
8,Syria,1969,0.0,0.0,0.0,14.990062
9,Syria,1970,0.0,0.0,0.0,-4.958800


# Step 3: Plot the relationship between the variables
In this step, I will plot the relatinoship between the intensity level of conflict, the cumulative intensity level and the type of conflict with the growth of GDP per capita in Syria seperately. <p>
Explanation of the code meanings:<br>
Source: UCDP/PRIO Armed Conflict Dataset version 25.1 Codebook
https://ucdp.uu.se/downloads/ucdpprio/ucdp-prio-acd-251.pdf 
1. intensity_level: The intensity level of the conflict in a given calender year. This variable is coded into two categories - <br>
    1= Minor: between 25 and 999 battle-related deaths in a given year. <br>
    2= War: at least 1,000 battle-related deaths in a given year <p>
2. cumulative_intensity: This variable is a dummy variable that codes whether the conflict since the onset has exceeded 1,000 battle-related deaths.<br>
    0: The conflict has not over time resulted in more than 1,000 battle-related deaths.<br>
    1: The conflict has reached the threshold of 1,000 battle-related deaths. <p>
3. type_of_conflict<br>
    1= extrasystemic (between a state and a non-state group outside its own territory, where the government side is fighting to retain control of a territory outside the state system)<br>
    2= interstate (both sides are states in the Gleditsch and Ward membership system).<br>
    3= intrastate (side A is always a government; side B is always one or more rebel groups; there is no involvement of foreign governments with troops, i.e. there is no side_a_2nd or side_b_2nd coded)<br>
    4= internationalized intrastate (side A is always a government; side B is always one or more rebel groups; there is involvement of foreign governments with troops, i.e. there is at least ONE side_a_2nd or side_b_2nd coded)


In [82]:
import plotly.express as px

# plot the relations between conflict intensity level and GDP per capita growth
fig1 = px.scatter(
    Merged_Data,
    x="intensity_level",
    y="GDP per capita",
    title="Figure 1: Relationship between Conflict Intensity Level and GDP per capita Growth in Syria (1961-2023)",
    labels={
        "intensity_level": "Conflict Intensity Level",
        "GDP per capita": "GDP per capita Growth",
    },
)
fig1.show()

In [85]:
# plot the relations between conflict cumulative intensity and GDP per capita growth
fig2 = px.scatter(
    Merged_Data,
    x="cumulative_intensity",
    y="GDP per capita",
    title="Figure 2: Relationship between Conflict Cumulative Intensity Level and GDP per capita Growth in Syria (1961-2023)",
    labels={
        "cumulative_intensity": "Conflict Cumulative Intensity Level",
        "GDP per capita": "GDP per capita Growth",
    },
)
fig2.show()

In [87]:
# plot the relations between conflict types and GDP per capita growth
fig3 = px.scatter(
    Merged_Data,
    x="type_of_conflict",
    y="GDP per capita",
    title="Figure 3: Relationship between Conflict Type and GDP per capita Growth in Syria (1961-2023)",
    labels={
        "type_of_conflict": "Conflict Type",
        "GDP per capita": "GDP per capita Growth",
    },
)
fig3.show()

# Takeaway
I am interested in exploring the relationship between conflict and economic development, which is why I choose to use the indicators, including conflict intensity level and annual GDP per capita growth. <p>
1. Analysis of visualization outcomes: <br>
(1) The visualization proved the first hypothesis in general. <br>
(2) Figure 1 demonstrates a negative correlation between conflict occurence & intensity and the growth of GDP per capita. The occurence of conflict in the given year is correlated with more negative GDP per capita growth. The higher intensity of conflicts is also correlated with more negative and lower GDP growth although the correlation is not as strong as that between the presence of conflicts in general and GDP per capita growth.<br>
(3) Figure 2 shows a negative correlation between ccumulative intensity of conflicts and the GDP per capita growth. There are more negative GDP per capita growth in those years when battle-related deaths exceed the threshold of 1000 and more positive growth when the deaths are below 1000 over time. However, the result could be different if we take into consideration of the compound effects that take place only in the given calendar year that could have a significant impact on the GDP per capita growth.<br>
(4) Figure 3 demonstrates the distribution of conflict types in Syria between 1961 and 2023 and the relationship between conflict type and GDP per capita growth. There is no significant correlation reflected by the visualization due to the uneven distribution of the conflict types and might require deeper analysis.<p>
2. Questions:<br>
(1) Is scatter plot the best way to reflect the relationship? I chose to use scatter plot because there is a significant number of NaN (later converted to 0) values in the conflict dataset, however it is unable to reflect the temporal variance of the variables. The incontinuous distribution of the scatter plot also means more reliance on observation than on numeric comparison.<br>
(2) Is there any way to show the correlation between conflict types and GDP per capita growth without resorting to further analysis like regression?


# Thank you for reading!